-   Student name, e-mail
-   Student name, e-mail

# Instructions for today's practice

- Create a copy of this Jupyter Notebook and share it with your partner.
- Fill student names and e-mails in the text cell above.
- At the end of the practice, SHARE THE NOTEBOOK LINK in the Moodle Submission
  - Make sure it is set up to "Public - Anyone with the link can view".


**Thanks to prof. Ken Arnold for writing this practice.**

# SLOs covered

In this practice, you will show your mastery of the following SLOs:
- WRA05	I can reshape data using melt and pivot to move between wide and long formats.
- WRA06	I can identify and use primary keys to connect related tables.
- WRA07	I can join dataframes using different join types (inner, left, right, outer).
- WRA08	I can identify how wrangling operations like grouping, reshaping and joining data may simplify or distort the underlying phenomena.

![](https://images.unsplash.com/photo-1502920514313-52581002a659?q=80&w=2067&auto=format&fit=crop&ixlib=rb-4.0.3&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D)

# Dataset: Gapminder stats

[Gapminder](https://www.gapminder.org/data/) is a Swedish foundation whose goal is to "promote sustainable global development through the use of data and statistics". It is best known for its interactive visualizations that help illustrate global trends in areas such as health, income, population, and development.

We will be downloading data from their website and use it to practice reshaping and join operations with pandas.

Step by step instructions:

- Go to https://www.gapminder.org/data/
- Under “Choose individual indicators”, search for “GDP”. Click the one marked “Income”, it should say “GDP per capita (price and inflation adjusted, in PPP$2017)” when you click on it.
- At the top of the table, click Download As: CSV or XLSX. I got a file named lex.xlsx
- Click the header (“GDP per capita”) and select instead “Life Expectancy, at birth”. Download that as well. I got a file named gdp_pcap.xlsx

Now, import these xlsx files and load the data:

In [ ]:
import pandas as pd

In [ ]:
gapminder_gdp = pd.read_excel("gdp_pcap.xlsx")
gapminder_gdp.head()

In [ ]:
gapminder_life = pd.read_excel("lex.xlsx")
gapminder_life.head()

# From wide to long (melt)

Notice that the data is in “wide” format, with each year as a column. To join and plot this data, we’ll need it in “long” format, with each year as a row.

We'll use `pd.melt` to get it in this format. We'll need to tell it which columns form the primary key, specified by the `id_vars` argument. The other columns will be “melted” into a single column, specified by the `var_name` argument. The values for those columns will be put in a column specified by the `value_name` argument.

**📝Task**: Fill in the blanks below with the appropriate column names. Then assign the result to a variable; I used `gdp_long`.

In [ ]:
gdp_long = pd.melt(
    gapminder_gdp,
    id_vars=[],
    var_name="XXX", value_name="YYY"
)

Now, repeat it for the life expectancy data (`life_long`).

In [ ]:
life_long = pd.melt(
    gapminder_life,
    id_vars=[...],
    var_name="XXX", value_name="YYY"
)

# Adjusting data types

Check the data types of the gdp_long table. There are two that are not what you’d expect.

In [ ]:
gdp_long.info()

Colum `year` is easy to fix:

In [ ]:
gdp_long['year'] = gdp_long['year'].astype(int)
life_long['year'] = life_long['year'].astype(int)

A more tricky issue is that gdp_pcap uses k to mean “thousands”. For example:

In [ ]:
gdp_long.tail()

We need to convert those k values to numbers. Here’s an approach:

In [ ]:
def parse_number_with_units(num):
    if not isinstance(num, str):
        return num
    if num.endswith("k"):
        return float(num[:-1]) * 1000
    return float(num)

gdp_long['gdp_pcap'] = gdp_long['gdp_pcap'].map(parse_number_with_units)
gdp_long.tail()

**📝Task**: Search for the documentation about the `map()` function in `pandas`, which we used here, and try to answer: **what exactly does it do?** Describe it succintly. This function is very powerful and useful for wrangling! If you have more questions, we can talk about it later.

*---your answer here---*

# Joining

**📝Task**: Now, join the datasets together. Call the result `gapminder`. This should have GDP and life expectancy of each country in each year.

Check for the total rows. You should have:
195 countries, 301 years = 58695 expected rows

# Adding regions to countries

Suppose we now want to visualize countries by region (Africa, Asia, etc). We need to perform another join, with another table!

The region data is available [here](https://www.gapminder.org/fw/four-regions/). `read_excel` can pull the data in directly for you. I'll save you some time by giving you that code:

In [ ]:
region_data = pd.read_excel("https://docs.google.com/spreadsheets/d/1qHalit8sXC0R8oVXibc2wa2gY7bkwGzOybEMTWp-08o/export?format=xlsx", sheet_name="list-of-countries-etc")
region_data.head()

**📝Task**: We only want the column `four_regions`. So, rename it to `region`. Also, rename the `name` column so it has the same name as the corresponding column in the `gapminder` dataframe. To rename, use `rename(columns={"old_name": "new_name"})`.

You can also drop all the other columns with `region_data = region_data[["country", "region"]]`

**📝Task**: Now, join `region_data` with the `gapminder` data, and assign the final result to a variable (I used `gapminder_with_regions`).

# Plot results



**📝Task**: Let's now plot our data using plotly. Import the library and use a scatter plot with the following mappings:
- **x axis** = gdp per capita
- **y axis** = life expectancy
- **color** = region (Asia, America, etc)
- **animation_frame** = year (this will create an interactive slider)

Also, add the following arguments to your `px.scatter` function:
- `log_x=True` - this will set the x axis to logarithmic scale
- `range_x=[100,100000], range_y=[25,90]` - set the ranges of the x and y axes
- `hover_name="country"` - this will show the name of the country when you hover a data point
- `labels={"gdpPercap": "GDP per capita", "lifeExp": "Life expectancy (at birth)", "region": "region", "year": "Year"}` - set the labels
- `title="Life expectancy vs. GDP per capita, 1952-2007"` - set the title

**📝Task**: With at least 3 sentences, explain how these wrangling choices (grouping, reshaping, joining) changed the way the data looked and what you could conclude. Mention at least one way these operations made the data easier to analyze, and at least one way they may have hidden or distorted important details. Conclude with how this awareness will affect how you interpret wrangled data in the future.